# Model Interpretability — Explaining a Black-Box Tabular Model

A gradient-boosted tree ensemble is a *black box*: it can be extremely accurate, yet its hundreds of trees give no single, readable rule for *why* it predicted what it did. This notebook opens that box two ways.

1. **Global interpretability with SHAP** — which features matter across the *whole* dataset? (What did the model learn overall?)
2. **Local interpretability with LIME** — for *one specific prediction*, which features pushed it toward its answer?

We use the **breast-cancer** dataset because its 30 features have real, human-readable names (`mean radius`, `worst concavity`, ...), so an explanation actually means something.

**Read this first — the notebook is offline-safe.** `xgboost`, `shap`, and `lime` may not be installed and there is no network to fetch them. *Every* use of them below is wrapped in `try / except ImportError` with a fully-working fallback built from packages you already have (`scikit-learn`, `numpy`). Each cell prints which path — real library vs. fallback — actually ran, so you always know what you are looking at.

## Optional: install the real libraries

If you have a network connection and want the genuine SHAP / LIME / XGBoost paths (instead of the fallbacks), run these once in a terminal or a notebook cell:

```bash
pip install shap
pip install lime
pip install xgboost
```

You do **not** need them to run this notebook — the fallbacks reproduce the same ideas with `scikit-learn`.

In [ ]:
import numpy as np                     # arrays, perturbation noise, weighted least squares
import pandas as pd                    # tidy feature tables keyed by real feature names
import matplotlib.pyplot as plt        # all plotting

from sklearn.datasets import load_breast_cancer      # small, offline, real feature names
from sklearn.model_selection import train_test_split # train/test split
from sklearn.metrics import accuracy_score           # trust-the-model check

# One global seed so every random step below (split, perturbations, model init) is reproducible.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("core scientific stack imported")

## 1. Data

`load_breast_cancer` returns 569 tumours described by 30 numeric features, each labelled **malignant (0)** or **benign (1)**. We keep the data in a `pandas.DataFrame` so every column carries its real name — the whole point of an *interpretable* example is that `worst radius` reads better than `feature_20`.

In [ ]:
# Load the bundled dataset (no download). as_frame=True gives us named columns for free.
data = load_breast_cancer(as_frame=True)
X = data.data                       # (569, 30) DataFrame; columns are the human-readable names
y = data.target                     # (569,) Series of 0/1 labels
feature_names = list(X.columns)     # e.g. 'mean radius', 'worst concave points', ...
class_names = [str(c) for c in data.target_names]  # ['malignant', 'benign']

# 75/25 split. stratify=y keeps the malignant/benign ratio identical in train and test.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"features: {len(feature_names)}   classes: {class_names}")
print(f"train: {X_train.shape}   test: {X_test.shape}")
X_train.head(3)   # peek at a few rows so the feature names are concrete

## 2. Train the black box (accuracy first)

Before we try to *explain* a model we must first *trust* it — an explanation of a bad model is worthless. We train a gradient-boosted tree ensemble and report test accuracy.

**Preferred:** `xgboost.XGBClassifier` (the tool most people mean by "the black box").

**Fallback:** `sklearn.ensemble.GradientBoostingClassifier` — the same *family* of algorithm (additive boosting of shallow trees), just scikit-learn's own implementation. Whichever loads, the rest of the notebook treats it identically as an opaque `.predict` / `.predict_proba` box.

In [ ]:
# Try the real XGBoost first; fall back to sklearn's gradient boosting if it isn't installed.
try:
    import xgboost as xgb                                        # not installed here -> ImportError
    model = xgb.XGBClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1,
        random_state=RANDOM_STATE, eval_metric="logloss",
    )
    MODEL_BACKEND = "xgboost.XGBClassifier (real library)"
except ImportError:
    # FALLBACK: same idea (boosted shallow trees), shipped inside scikit-learn.
    from sklearn.ensemble import GradientBoostingClassifier
    model = GradientBoostingClassifier(
        n_estimators=200, max_depth=3, learning_rate=0.1, random_state=RANDOM_STATE
    )
    MODEL_BACKEND = "sklearn.GradientBoostingClassifier (FALLBACK -- xgboost not installed)"

print(f"[black box] using: {MODEL_BACKEND}")

# Fit on the training split only.
model.fit(X_train, y_train)

# Trust check: how accurate is it on data it never saw during training?
test_acc = accuracy_score(y_test, model.predict(X_test))
print(f"[black box] test accuracy: {test_acc:.3f}")
print("High accuracy -> the explanations below are worth interpreting.")

## 3. Global interpretability with SHAP

**Global** interpretability answers *"across the whole dataset, which features drive this model?"* — one ranking that summarizes the model's overall behaviour.

### The intuition behind SHAP (Shapley values)
SHAP borrows the **Shapley value** from cooperative game theory. Imagine the features are *players* cooperating to produce a prediction, and the prediction (relative to a baseline) is the *payout* to be split fairly among them. A feature's Shapley value is its **average marginal contribution over every possible coalition** (subset) of the other features:

$$\phi_i = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!\,(|F| - |S| - 1)!}{|F|!}\,\big[\, f(S \cup \{i\}) - f(S) \,\big]$$

where $F$ is all features, $S$ a coalition without feature $i$, and $f(S)$ the model's expected output using only the features in $S$. The term in brackets is *"how much did adding feature $i$ change the prediction?"*, averaged fairly over all orderings. This is the unique attribution satisfying **efficiency** (the $\phi_i$ sum to the prediction minus the baseline), **symmetry**, and **null-player** fairness axioms.

For a global view we take the **mean absolute SHAP value** of each feature, $\text{mean}_j |\phi_i^{(j)}|$ over all rows $j$ — big mean $|\phi_i|$ means the feature moves predictions a lot, in either direction.

**Preferred:** `shap.TreeExplainer` computes exact tree Shapley values efficiently.

**Fallback:** `sklearn.inspection.permutation_importance` — shuffle one feature's column and measure how much accuracy drops. This is a *proxy* for SHAP's global view: both rank features by overall influence, but permutation importance measures *impact on accuracy* rather than a fair, additive credit split, and it can behave oddly when features are correlated. It gives the same *shape* of answer (a global ranking), not the same guarantees.

In [ ]:
# Compute a GLOBAL feature ranking, preferring SHAP and falling back to permutation importance.
try:
    import shap                                          # not installed here -> ImportError
    # TreeExplainer gives exact Shapley values for tree ensembles.
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_test)
    # For binary classifiers some SHAP versions return a list [class0, class1]; take the
    # positive class if so, otherwise use the array directly.
    sv = shap_values[1] if isinstance(shap_values, list) else shap_values
    global_importance = np.abs(sv).mean(axis=0)          # mean|SHAP| per feature
    GLOBAL_BACKEND = "shap.TreeExplainer -- mean|SHAP| (real library)"
    global_xlabel = "mean(|SHAP value|)"
except ImportError:
    # FALLBACK: permutation importance as a PROXY for SHAP's global ranking.
    from sklearn.inspection import permutation_importance
    # Shuffle each feature 10x; the mean accuracy drop is that feature's importance.
    perm = permutation_importance(
        model, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE
    )
    global_importance = perm.importances_mean            # mean accuracy drop per feature
    GLOBAL_BACKEND = "permutation_importance (FALLBACK -- proxy for SHAP's global view)"
    global_xlabel = "mean accuracy drop when shuffled"

print(f"[global] using: {GLOBAL_BACKEND}")

# Rank features and keep the top 10 for a readable bar chart.
order = np.argsort(global_importance)[::-1]              # indices, most important first
top_k = 10
top_idx = order[:top_k]
top_feats = [feature_names[i] for i in top_idx]
top_vals = global_importance[top_idx]

plt.figure(figsize=(8, 5))
# Horizontal bars, largest at the top (invert so rank 1 sits on top).
plt.barh(top_feats[::-1], top_vals[::-1], color="#4c72b0")
plt.xlabel(global_xlabel)
plt.title(f"Global feature importance (top {top_k})\n{GLOBAL_BACKEND}")
plt.tight_layout()
plt.show()

print("\nTop 5 globally important features:")
for rank, i in enumerate(top_idx[:5], 1):
    print(f"  {rank}. {feature_names[i]:<28} {global_importance[i]:.4f}")

## 4. Local interpretability with LIME

**Local** interpretability zooms all the way in: *"for this **one** patient, which features pushed the model toward its prediction?"* The global ranking tells you what the model cares about *on average*; a local explanation can look completely different for an individual case.

### The intuition behind LIME (local linear surrogate)
LIME = **L**ocal **I**nterpretable **M**odel-agnostic **E**xplanations. The black box may be wildly nonlinear *globally*, but *right next to a single point* almost any smooth function looks roughly linear. LIME exploits that:

1. Take the instance $x$ you want to explain.
2. **Perturb** it many times (add noise / sample nearby points) to get a cloud of synthetic neighbours.
3. Ask the **black box** for its prediction on each neighbour — these are the labels the surrogate must match.
4. **Weight** each neighbour by how close it is to $x$ (near points matter more than far ones).
5. Fit a simple **weighted linear model** to those labelled neighbours.
6. Read off the linear model's **coefficients** — each is that feature's local contribution to this one prediction.

Formally LIME minimizes a locality-weighted fidelity loss plus a simplicity penalty:
$$\xi(x) = \arg\min_{g \in G} \; \mathcal{L}\big(f, g, \pi_x\big) + \Omega(g)$$
where $f$ is the black box, $g$ the simple surrogate, $\pi_x$ the proximity weighting around $x$, and $\Omega$ favours sparse/simple $g$.

**Preferred:** `lime.lime_tabular.LimeTabularExplainer`.

**Fallback:** we implement steps 1–6 **by hand** with NumPy + a weighted `LinearRegression`. This is not a toy stand-in — it *is* the core algorithm behind LIME, just written out explicitly so you can see every moving part.

In [ ]:
# Pick ONE test instance to explain locally.
instance_pos = 0                                  # first row of the test set
x_instance = X_test.iloc[instance_pos]            # the row as a named Series
x_row = x_instance.values.astype(float)           # bare numpy vector, shape (30,)

# What did the black box actually predict for this one patient?
pred_class = int(model.predict(x_instance.to_frame().T)[0])
pred_proba = model.predict_proba(x_instance.to_frame().T)[0]
true_class = int(y_test.iloc[instance_pos])

print(f"Explaining test instance #{instance_pos}")
print(f"  predicted class : {pred_class} ({class_names[pred_class]})")
print(f"  P(malignant)={pred_proba[0]:.3f}  P(benign)={pred_proba[1]:.3f}")
print(f"  true class      : {true_class} ({class_names[true_class]})")

In [ ]:
# Produce a LOCAL explanation for x_instance, preferring LIME, falling back to a hand-built
# local linear surrogate. Both yield (feature_name, signed_contribution) pairs.
try:
    import lime                                          # not installed here -> ImportError
    import lime.lime_tabular
    lime_explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=X_train.values,
        feature_names=feature_names,
        class_names=class_names,
        discretize_continuous=True,
        random_state=RANDOM_STATE,
    )
    exp = lime_explainer.explain_instance(
        x_row, model.predict_proba, num_features=10, labels=(pred_class,)
    )
    # exp.as_list() gives ('feature condition', weight) for the explained label.
    local_pairs = exp.as_list(label=pred_class)
    local_feats = [name for name, _ in local_pairs]
    local_contribs = np.array([w for _, w in local_pairs])
    LOCAL_BACKEND = "lime.lime_tabular (real library)"
except ImportError:
    # FALLBACK: the LIME algorithm written out by hand (this IS how LIME works).
    from sklearn.linear_model import LinearRegression
    rng = np.random.default_rng(RANDOM_STATE)

    # (1) locality scale: use each feature's TRAIN std so noise respects feature ranges.
    feat_std = X_train.values.std(axis=0) + 1e-12

    # (2) PERTURB: draw 1000 synthetic neighbours around x by adding Gaussian noise.
    n_samples = 1000
    noise = rng.normal(0.0, 1.0, size=(n_samples, x_row.shape[0]))
    perturbed = x_row + noise * feat_std                 # (1000, 30) neighbours in real units
    perturbed[0] = x_row                                 # keep the instance itself as one sample

    # (3) BLACK-BOX LABELS: probability of the predicted class for every neighbour.
    perturbed_df = pd.DataFrame(perturbed, columns=feature_names)
    y_surrogate = model.predict_proba(perturbed_df)[:, pred_class]

    # (4) WEIGHTS: closeness of each neighbour to x via an RBF (exponential) kernel on the
    #     standardized distance -> near points get weight ~1, far points ~0.
    z = (perturbed - x_row) / feat_std                   # standardized offsets
    dist = np.sqrt((z ** 2).sum(axis=1))                 # Euclidean distance in std units
    kernel_width = np.sqrt(x_row.shape[0]) * 0.75        # standard LIME default scale
    weights = np.exp(-(dist ** 2) / (kernel_width ** 2)) # proximity weights, shape (1000,)

    # (5) FIT a weighted linear surrogate on STANDARDIZED features (z) so coefficients are
    #     comparable across features regardless of their raw scale.
    surrogate = LinearRegression()
    surrogate.fit(z, y_surrogate, sample_weight=weights)

    # (6) READ OFF coefficients = each feature's local contribution; keep the 10 largest |coef|.
    coefs = surrogate.coef_
    top_local = np.argsort(np.abs(coefs))[::-1][:10]
    local_feats = [feature_names[i] for i in top_local]
    local_contribs = coefs[top_local]
    LOCAL_BACKEND = "hand-built local linear surrogate (FALLBACK -- this IS the LIME algorithm)"

print(f"[local] using: {LOCAL_BACKEND}")
for name, c in zip(local_feats, local_contribs):
    direction = f"-> {class_names[pred_class]}" if c > 0 else f"-> {class_names[1 - pred_class]}"
    print(f"  {name:<32} {c:+.4f}  {direction}")

In [ ]:
# Horizontal bar chart of the per-feature LOCAL contributions for this single prediction.
# Green bars push toward the predicted class; red bars push away from it.
order_local = np.argsort(np.abs(local_contribs))         # smallest |contrib| first
feats_sorted = [local_feats[i] for i in order_local]
contribs_sorted = local_contribs[order_local]
colors = ["#2a9d8f" if c > 0 else "#e76f51" for c in contribs_sorted]

plt.figure(figsize=(8, 5))
plt.barh(feats_sorted, contribs_sorted, color=colors)
plt.axvline(0, color="k", linewidth=0.8)                 # zero line = no local influence
plt.xlabel(f"local contribution toward '{class_names[pred_class]}'")
plt.title(
    f"LOCAL explanation for test instance #{instance_pos}"
    f"  (pred: {class_names[pred_class]})\n{LOCAL_BACKEND}"
)
plt.tight_layout()
plt.show()

print("Green = pushed the prediction toward the predicted class; Red = pushed away.")

## 5. Global vs. local, and the caveats

| | **SHAP (global view here)** | **LIME (local view here)** |
|---|---|---|
| **Question** | Which features drive the model *overall*? | Why *this one* prediction? |
| **Scope** | Whole dataset (one ranking) | A single instance |
| **Mechanism** | Fair Shapley credit split over feature coalitions | Weighted linear surrogate fit near the instance |
| **Output here** | mean\|SHAP\| bar chart (or permutation-importance proxy) | signed per-feature contribution bars |

**Why you need both.** A globally dominant feature (say `worst perimeter`) might contribute almost nothing to a *particular* patient whose value is unremarkable, while a normally-minor feature dominates that one case. Global tells you what the model learned; local tells you what happened *here*.

### Caveats — read before you trust any explanation
- **SHAP / Shapley values.** Exact Shapley computation is exponential in the number of features; `TreeExplainer` is fast *only* because trees have special structure. General (kernel) SHAP approximates, and — like most methods — can be misleading when features are strongly **correlated**, because it must assume how to "remove" a feature.
- **Permutation importance (our SHAP fallback).** It measures *drop in accuracy when a column is shuffled*, not a fair additive attribution. Correlated features can share/mask importance, and shuffling creates unrealistic feature combinations. Treat it as a *rough proxy* for the global ranking, not as Shapley values.
- **LIME / local surrogates.** Explanations depend on the **perturbation scheme and kernel width** — change the neighbourhood and the coefficients can change. The linear surrogate is only faithful in a *small* region around the instance; it says nothing about the model elsewhere. Results can also be **unstable** run-to-run because of the random sampling.
- **All of them are descriptive, not causal.** "Feature X pushed the prediction up" is a statement about *the model*, not about biology. An explanation can faithfully reflect a model that learned a spurious correlation.

Interpretability tools tell you what your model is *doing* — a necessary first step, but always sanity-check against domain knowledge before acting on them.